**Global Cancer Patients 데이터셋**

2015년부터 2024년까지 보고된 전 세계 암 환자 데이터를 포함하고 있으며, 암의 진단, 치료, 생존에 영향을 미치는 주요 요인들을 시뮬레이션한 데이터입니다.

제공된 학습용 데이터(cacner_train.csv)를 이용하여 암의 심각도(Severity)를 예측하는 모델을 개발하고, 개발한 모델에 기반하여 평가용 데이터(cancer_test.csv)에 적용하여 얻은 암의 심각도 예측 값을 아래 [제출형식]에 따라 csv 파일로 생성하여 제출하시오.
- 예측 결과는 F1 Score(macro) 평가지표에 따라 평가함
- 성능이 우수한 예측 모델을 구축하기 위해서는 데이터 정제, Feature Engineering, 하이퍼 파라미터(hyper parameter) 최적화, 모델 비교 등이 필요할 수 있음. 다만, 과적합에 유의하여야 함


[[제출 형식]]
- 가. CSV 파일명: result.csv(파일명에 디렉토리/폴더 지정불가
- 나. 예측 칼럼명 : pred
- 다. 제출 칼럼 개수 : pred 칼럼 1개
- 라. 평가용 데이터 개수와 예측 결과 데이터 개수 일치 : 2,193개

[[제공 데이터]]
- 데이터 목록
- cancer_train.csv : 학습용 데이터, 5,115개
- cancer_test.csv : 평가용 데이터, 2,193개
- 평가용 데이터는 'Severity' 칼럼 미제

In [30]:
# 라이브러리
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import f1_score
pd.set_option('display.width',120)

# 데이터 불러오기
path = "https://raw.githubusercontent.com/Soyoung-Yoon/bigdata/main/"
train = pd.read_csv(path + "cancer_train.csv")
test = pd.read_csv(path + "cancer_test.csv")
# print(train.head(3), train.shape, sep="\n")
# print(test.head(3), test.shape, sep="\n")

# 데이터 전처리
X = train.drop(columns=['Patient_ID', 'Severity'])
Y = train['Severity']
X_submission = test.drop(columns=['Patient_ID'])
X_all = pd.concat([X, X_submission])
cols_obj = X_all.select_dtypes(include='object').columns
for col in cols_obj:
    X_all[col] = LabelEncoder().fit_transform(X_all[col])
# print(X_all.head(3))
X = X_all.iloc[:len(X), :]
X_submission = X_all.iloc[len(X):, :]
# print(X.shape, X_submission.shape) # (5115, 13) (2193, 13)
temp = train_test_split(X, Y, test_size=0.3, stratify=Y, random_state=123)
x_train, x_test, y_train, y_test = temp
# print(x_train.shape, x_test.shape, y_train.shape, y_test.shape) # (3580, 13) (1535, 13) (3580,) (1535,)

# 파이프라인 모델사전
models = {
    "Logistic": Pipeline([
        ('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=500, tol=0.05, random_state=123))
    ]),
    "DecisionTree": Pipeline([
        ('model', DecisionTreeClassifier(max_depth=3, random_state=123))
    ]),
    "RandomForest": Pipeline([
        ('model', RandomForestClassifier(max_depth=3, random_state=123))
    ]),
    "AdaBoost": Pipeline([
        ('model', AdaBoostClassifier(n_estimators=500, random_state=123))
    ]),
    "GradientBoosting": Pipeline([
        ('model', GradientBoostingClassifier(random_state=123))
    ])
}

# 성능평가 함수
def get_scores(model, x_train, x_test, y_train, y_test):
    model.fit(x_train, y_train)
    y_pred1 = model.predict(x_train)
    y_pred2 = model.predict(x_test)
    f1_train = f1_score(y_train, y_pred1, average="macro")
    f1_test = f1_score(y_test, y_pred2, average="macro")
    return model, f1_train, f1_test

# 평가결과
results = []
for name, model in models.items():
    model, f1_train, f1_test = get_scores(model, x_train, x_test, y_train, y_test)
    results.append({
        "Model": name, "F1_train": round(f1_train, 3), "F1_test": round(f1_test, 3)
    })

res = pd.DataFrame(results).sort_values("F1_test", ascending=False).reset_index(drop=True)
print(res)

# 모델학습
model = models[res.loc[0, 'Model']]
y_pred = model.predict(X_submission)
# print(y_pred)

# 결과생성
pd.DataFrame({'pred': y_pred}).to_csv("result_type2_2th.csv", index=False)

# 결과확인
temp = pd.read_csv("result_type2_2th.csv")
print(temp['pred'].value_counts(normalize=True))
print("=" * 3)
print(Y[:len(X_submission)].value_counts(normalize=True))


              Model  F1_train  F1_test
0          Logistic     0.948    0.947
1  GradientBoosting     0.990    0.944
2          AdaBoost     0.867    0.848
3      RandomForest     0.849    0.827
4      DecisionTree     0.695    0.676
pred
High      0.369357
Medium    0.363429
Low       0.267214
Name: proportion, dtype: float64
===
Severity
Medium    0.411309
High      0.324669
Low       0.264022
Name: proportion, dtype: float64
